In [ ]:
# -------------------------------------------------------------
# Cell 1: Install Dependencies
# -------------------------------------------------------------
# Fix numpy/scipy version mismatch (scipy 1.14+ requires numpy >= 2.0)
!pip install -q -U numpy scipy
!pip install -q fastapi python-multipart uvicorn nest_asyncio pyngrok
!pip install -q whisperx praat-parselmouth opensmile audiofile
!pip install -q nltk
!python -m nltk.downloader averaged_perceptron_tagger_eng
import IPython
IPython.Application.instance().kernel.do_shutdown(True)  # Restart kernel after upgrades


In [ ]:
# -------------------------------------------------------------
# Cell 2: API Server Startup & Execution
# -------------------------------------------------------------
import os
import json
import time
import tempfile
import subprocess
import numpy as np
import torch
# -- Patch: fix PyTorch 2.6 weights_only=True default breaking trusted model loading --
if not hasattr(torch, "_orig_load"):
    torch._orig_load = torch.load

def _patched_torch_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return torch._orig_load(*args, **kwargs)

torch.load = _patched_torch_load
# -- End patch --
import torchaudio
# -- Patch: fix torchaudio / pyannote compatibility issue in newer torchaudio versions --
if not hasattr(torchaudio, "AudioMetaData"):
    class DummyAudioMetaData:
        pass
    torchaudio.AudioMetaData = DummyAudioMetaData

if not hasattr(torchaudio, "list_audio_backends"):
    def dummy_list_audio_backends():
        return ["soundfile", "ffmpeg", "sox"]
    torchaudio.list_audio_backends = dummy_list_audio_backends
# -- End patch --
import parselmouth
from parselmouth.praat import call
import whisperx
import nltk
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException
from fastapi.responses import JSONResponse
from typing import List, Dict, Any, Tuple
import uvicorn
import nest_asyncio
from pyngrok import ngrok

nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

# -- Patch: fix faster-whisper / ctranslate2 incompatibility on this env --
import ctranslate2
if not hasattr(ctranslate2.models, "_orig_whisper"):
    ctranslate2.models._orig_whisper = ctranslate2.models.Whisper

class _PatchedWhisper(ctranslate2.models._orig_whisper):
    def __init__(self, *args, **kwargs):
        kwargs.pop("use_auth_token", None)
        super().__init__(*args, **kwargs)

ctranslate2.models.Whisper = _PatchedWhisper
# -- End patch --

# -------------------------------------------------------------
# A. Initialize ASR & Audio Processing Models
# -------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if "asr_model" not in globals():
    print(f"[startup] Loading WhisperX large-v3 model on {DEVICE}...")
    try:
        # Use Silero VAD (lightweight, avoids pyannote TF32 reproducibility crashes)
        asr_model = whisperx.load_model(
            "large-v3",
            DEVICE,
            compute_type="float16" if DEVICE == "cuda" else "int8",
            language="en",
            vad_method="silero",
            asr_options={
                "initial_prompt": "Um, uh, like, you know, I mean, er, ah, so...",
                "suppress_tokens": [],
                "beam_size": 8
            }
        )
        print("[startup] ASR model loaded with Silero VAD.")
    except TypeError:
        # Older whisperx version - fall back to default (pyannote) VAD
        asr_model = whisperx.load_model(
            "large-v3",
            DEVICE,
            compute_type="float16" if DEVICE == "cuda" else "int8",
            language="en",
            asr_options={
                "initial_prompt": "Um, uh, like, you know, I mean, er, ah, so...",
                "suppress_tokens": [],
                "beam_size": 8
            }
        )
        print("[startup] ASR model loaded with default VAD (vad_method not supported).")
else:
    print("[startup] WhisperX large-v3 model already loaded in globals.")

if "align_model" not in globals():
    print("[startup] Loading WhisperX English alignment model...")
    align_model, align_metadata = whisperx.load_align_model(language_code="en", device=DEVICE)
else:
    print("[startup] WhisperX English alignment model already loaded in globals.")

# -------------------------------------------------------------
# A2. Dual-Profile Behavioral Engine Classes
# -------------------------------------------------------------
class BehavioralProfile:
    def __init__(self, name: str):
        self.name = name
        self.features_history = []
        self.stats = {}

    def load_from_stats(self, stats: dict):
        self.stats = {}
        key_mappings = {
            "pitch_mean": ("pitch_mean", "mean"),
            "pitch_range": ("pitch_range", "mean"),
            "jitter": ("jitter", "mean"),
            "shimmer": ("shimmer", "mean"),
            "hnr": ("hnr", "mean"),
            "wps": ("wps_mean", "mean"),
            "gap_variance": ("gap_var_mean", "mean"),
            "ttr": ("ttr_mean", "mean"),
            "nvr": ("nvr_mean", "mean"),
            "cfr": ("cfr_mean", "mean"),
            "filler_rate": ("filler_mean", "mean"),
            "pitch_accel": ("pitch_accel", "mean"),
            "energy_fluctuation": ("energy_fluctuation", "mean"),
            "pause_entropy": ("pause_ent_mean", "mean")
        }
        
        for target_key, (src_key, stat_type) in key_mappings.items():
            val_mean = None
            val_std = 0.01
            
            if src_key in stats:
                val_mean = stats[src_key].get("mean")
                val_std = stats[src_key].get("std", 0.01)
            else:
                for sub_dict in stats.values():
                    if isinstance(sub_dict, dict) and src_key in sub_dict:
                        val_mean = sub_dict[src_key].get("mean")
                        val_std = sub_dict[src_key].get("std", 0.01)
                        break
                        
            if val_mean is None:
                if target_key in stats:
                    val_mean = stats[target_key].get("mean")
                    val_std = stats[target_key].get("std", 0.01)
                else:
                    for sub_dict in stats.values():
                        if isinstance(sub_dict, dict) and target_key in sub_dict:
                            val_mean = sub_dict[target_key].get("mean")
                            val_std = sub_dict[target_key].get("std", 0.01)
                            break
                            
            if val_mean is not None:
                self.stats[target_key] = {
                    "mean": float(val_mean),
                    "std": float(val_std) or 0.01
                }

    def update(self, feats: dict):
        self.features_history.append(feats)
        if len(self.features_history) > 50:
            self.features_history.pop(0)
            
        for key in feats.keys():
            vals = [h[key] for h in self.features_history if h.get(key) is not None]
            if len(vals) > 0:
                self.stats[key] = {
                    "mean": float(np.mean(vals)),
                    "std": float(np.std(vals)) or 0.01
                }

    def compute_similarity(self, feats: dict) -> float:
        if not self.stats:
            return 0.0
            
        sims = []
        for key, val in feats.items():
            if key in self.stats and val is not None:
                b_mean = self.stats[key]["mean"]
                b_std = self.stats[key]["std"]
                z = abs(val - b_mean) / (b_std + 1e-6)
                sim = float(np.exp(-0.5 * z ** 2))
                sims.append(sim)
        return float(np.mean(sims)) if sims else 0.0


class DualProfileEngine:
    def __init__(self, script_stats: dict, naturality_threshold: float = 0.45):
        self.script_profile = BehavioralProfile("SCRIPT")
        self.script_profile.load_from_stats(script_stats)
        
        self.natural_profile = BehavioralProfile("NATURAL")
        self.naturality_threshold = naturality_threshold
        
    def compute_naturality_score(self, feats: dict) -> float:
        filler_rate = feats.get("filler_rate", 0.0)
        self_corrections = feats.get("self_corrections", 0.0)
        retrieval_pauses = feats.get("retrieval_pauses", 0.0)
        pause_entropy = feats.get("pause_entropy", 0.0)
        pitch_accel = feats.get("pitch_accel", 0.0)
        energy_fluctuation = feats.get("energy_fluctuation", 0.0)
        
        f_fillers = min(1.0, filler_rate / 0.05)
        f_corrections = min(1.0, self_corrections / 1.0)
        f_pauses = min(1.0, retrieval_pauses / 2.0)
        f_entropy = min(1.0, pause_entropy / 1.5)
        f_pitch_accel = min(1.0, pitch_accel / 30.0)
        f_energy = min(1.0, energy_fluctuation / 0.3)
        
        naturality_score = (
            0.20 * f_fillers +
            0.15 * f_corrections +
            0.15 * f_pauses +
            0.15 * f_entropy +
            0.15 * f_pitch_accel +
            0.20 * f_energy
        )
        return float(naturality_score)
        
    def process_window(self, feats: dict) -> dict:
        script_similarity = self.script_profile.compute_similarity(feats)
        
        if len(self.natural_profile.features_history) >= 3:
            natural_similarity = self.natural_profile.compute_similarity(feats)
        else:
            natural_similarity = 0.0
            
        naturality_score = self.compute_naturality_score(feats)
        
        if naturality_score > self.naturality_threshold and script_similarity < 0.50:
            self.natural_profile.update(feats)
            
        raw_score = script_similarity - natural_similarity
        mapped_score = max(0.0, raw_score)
        
        return {
            "script_similarity": script_similarity,
            "natural_similarity": natural_similarity,
            "naturality_score": naturality_score,
            "gpu_raw_score": mapped_score
        }


class TransitionDetector:
    def __init__(self, window_size: int = 5):
        self.history = []
        self.window_size = window_size
        
    def update(self, script_similarity: float, naturality_score: float, feats: dict) -> dict:
        self.history.append({
            "script_similarity": script_similarity,
            "naturality_score": naturality_score,
            "feats": feats
        })
        if len(self.history) > self.window_size:
            self.history.pop(0)
            
        if len(self.history) < self.window_size:
            return {"transition_detected": False, "confidence": 0.0}
            
        half = self.window_size // 2
        first_half = self.history[:half]
        second_half = self.history[half:]
        
        avg_nat_before = np.mean([w["naturality_score"] for w in first_half])
        avg_nat_after = np.mean([w["naturality_score"] for w in second_half])
        nat_drop = avg_nat_before - avg_nat_after
        
        avg_script_before = np.mean([w["script_similarity"] for w in first_half])
        avg_script_after = np.mean([w["script_similarity"] for w in second_half])
        script_rise = avg_script_after - avg_script_before
        
        fillers_before = sum(w["feats"].get("filler_rate", 0.0) for w in first_half)
        fillers_after = sum(w["feats"].get("filler_rate", 0.0) for w in second_half)
        filler_drop = fillers_before > 0.01 and fillers_after < 0.01
        
        is_transition = False
        confidence = 0.0
        
        if nat_drop > 0.15 and script_rise > 0.15:
            is_transition = True
            confidence = float(min(1.0, (nat_drop + script_rise) / 2.0))
        elif filler_drop and script_rise > 0.12:
            is_transition = True
            confidence = float(min(1.0, 0.50 + script_rise))
            
        return {
            "transition_detected": is_transition,
            "confidence": confidence
        }

# -------------------------------------------------------------
# B. Parselmouth Feature Extraction Helper
# -------------------------------------------------------------
def extract_parselmouth_features(signal: np.ndarray, sr: int) -> dict:
    fallback = {
        "pitch_mean": 150.0,
        "pitch_range": 100.0,
        "jitter": 0.01,
        "shimmer": 0.05,
        "hnr": 12.0,
        "pitch_accel": 0.0,
        "energy_fluctuation": 0.0
    }
    if signal is None or len(signal) == 0:
        return fallback
    if np.isnan(signal).any() or np.isinf(signal).any():
        return fallback
    if np.max(np.abs(signal)) < 1e-4:
        return fallback
        
    try:
        sound = parselmouth.Sound(signal, sampling_rate=sr)
        
        # Extract pitch features
        pitch = call(sound, "To Pitch", 0.0, 75.0, 600.0)
        mean_pitch = call(pitch, "Get mean", 0.0, 0.0, "Hertz")
        min_pitch = call(pitch, "Get minimum", 0.0, 0.0, "Hertz", "Parabolic")
        max_pitch = call(pitch, "Get maximum", 0.0, 0.0, "Hertz", "Parabolic")
        pitch_range = max_pitch - min_pitch if (max_pitch != parselmouth.praat.NA and min_pitch != parselmouth.praat.NA) else 0.0
        
        # Point process for voice features (Jitter, Shimmer, HNR)
        point_process = call(sound, "To PointProcess (periodic, cc)", 75.0, 600.0)
        
        # Local Jitter
        local_jitter = call(point_process, "Get jitter (local)", 0.0, 0.0, 0.0001, 0.02, 1.3)
        
        # Local Shimmer
        local_shimmer = call([sound, point_process], "Get shimmer (local_db)", 0.0, 0.0, 0.0001, 0.02, 1.3, 1.6)
        
        # HNR (Harmonicity)
        harmonicity = call(sound, "To Harmonicity (cc)", 0.01, 75.0, 0.1, 4.5)
        hnr = call(harmonicity, "Get mean", 0.0, 0.0)
        
        # Extract pitch acceleration
        try:
            pitch_values = pitch.selected_array['frequency']
            voiced_pitches = pitch_values[pitch_values > 0]
            pitch_accel = 0.0
            if len(voiced_pitches) > 2:
                pitch_diffs = np.diff(voiced_pitches)
                pitch_accel = float(np.mean(np.abs(np.diff(pitch_diffs))))
        except Exception:
            pitch_accel = 0.0
            
        # Extract energy fluctuation
        try:
            intensity = sound.to_intensity()
            intensity_values = intensity.values[0]
            energy_fluctuation = 0.0
            if len(intensity_values) > 1:
                energy_fluctuation = float(np.std(intensity_values) / (np.mean(intensity_values) + 1e-6))
        except Exception:
            energy_fluctuation = 0.0

        res = {
            "pitch_mean": float(mean_pitch) if mean_pitch != parselmouth.praat.NA else 150.0,
            "pitch_range": float(pitch_range) if pitch_range != parselmouth.praat.NA else 100.0,
            "jitter": float(local_jitter) if local_jitter != parselmouth.praat.NA else 0.01,
            "shimmer": float(local_shimmer) if local_shimmer != parselmouth.praat.NA else 0.05,
            "hnr": float(hnr) if hnr != parselmouth.praat.NA else 12.0,
            "pitch_accel": pitch_accel,
            "energy_fluctuation": energy_fluctuation
        }
        return res
    except Exception as e:
        return fallback

# -------------------------------------------------------------
# C. Setup FastAPI Server & Security
# -------------------------------------------------------------
app = FastAPI(title="SentinEL Remote T4 GPU Server")

# Expected secret key (can be updated dynamically)
KAGGLE_SECRET = os.getenv("KAGGLE_SECRET", "sentinEL2026")

def validate_sentinel_secret(secret: str, x_sentinel_secret: str):
    provided = secret or x_sentinel_secret
    if KAGGLE_SECRET and provided != KAGGLE_SECRET:
        raise HTTPException(status_code=401, detail="Unauthorized: Invalid secret key.")

@app.get("/health")
async def health():
    return {"status": "ok", "device": DEVICE, "model_loaded": True}

# We define calibrate properly to support both audio_file and mode parameter
@app.post("/calibrate")
async def calibrate_endpoint(
    audio_file: UploadFile = File(None),
    mode: UploadFile = File(None),
    secret: str = Form(None),
    x_sentinel_secret: str = Header(None)
):
    validate_sentinel_secret(secret, x_sentinel_secret)
    file_obj = audio_file or mode
    if not file_obj:
        raise HTTPException(status_code=400, detail="No file uploaded.")
        
    t0 = time.perf_counter()
    tmp_path = None
    wav_path = None
    try:
        orig_ext = os.path.splitext(file_obj.filename or ".wav")[1] or ".wav"
        with tempfile.NamedTemporaryFile(delete=False, suffix=orig_ext) as tmp:
            tmp.write(await file_obj.read())
            tmp_path = tmp.name
            
        wav_path = tmp_path + "_cal.wav"
        # Use ffmpeg to convert to 16kHz mono wav
        ff_res = subprocess.run(
            ["ffmpeg", "-y", "-i", tmp_path, "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", wav_path],
            capture_output=True, text=True
        )
        if ff_res.returncode != 0:
            raise RuntimeError(f"ffmpeg audio extraction failed: {ff_res.stderr}")
                
        import audiofile
        audio_data, sr = audiofile.read(wav_path)
        if len(audio_data.shape) > 1:
            audio_data = np.mean(audio_data, axis=0)
            
        # 1. Parselmouth baseline stats (sliding 4s window with 2s hop)
        parselmouth_features = []
        hop_samples = int(2.0 * sr)
        win_samples = int(4.0 * sr)
        for start in range(0, len(audio_data) - win_samples, hop_samples):
            chunk = audio_data[start : start + win_samples]
            feats = extract_parselmouth_features(chunk, sr)
            parselmouth_features.append(feats)
            
        parselmouth_baseline = {}
        for key in ["pitch_mean", "pitch_range", "jitter", "shimmer", "hnr", "pitch_accel", "energy_fluctuation"]:
            vals = [f[key] for f in parselmouth_features] if parselmouth_features else [0.0]
            parselmouth_baseline[key] = {
                "mean": float(np.mean(vals)),
                "std": float(np.std(vals)) or 0.01
            }
            
        # 2. WhisperX baseline stats
        result = asr_model.transcribe(audio_data.astype(np.float32), batch_size=2, language="en")
        segments = result.get("segments", [])
        
        all_words_info = []
        if segments:
            aligned_result = whisperx.align(segments, align_model, align_metadata, audio_data.astype(np.float32), device=DEVICE, return_char_alignments=False)
            for seg in aligned_result.get("segments", []):
                for w in seg.get("words", []):
                    all_words_info.append(w)
                    
        timed_words = [w for w in all_words_info if "start" in w and "end" in w]
        
        # Calculate WPS, gap variance, and linguistic baseline metrics per 4-second window
        wps_vals = []
        gap_vars = []
        ttr_vals = []
        nvr_vals = []
        cfr_vals = []
        filler_rate_vals = []
        pause_ent_vals = []
        duration = len(audio_data) / sr
        for w_start in np.arange(0, duration - 4.0, 2.0):
            w_end = w_start + 4.0
            words_in_window = [w for w in timed_words if w_start <= w["start"] <= w_end]
            wps = len(words_in_window) / 4.0
            wps_vals.append(wps)
            
            gaps = []
            for j in range(len(words_in_window) - 1):
                gap = words_in_window[j+1]["start"] - words_in_window[j]["end"]
                if gap >= 0.0:
                    gaps.append(gap)
            gap_var = float(np.var(gaps)) if len(gaps) > 1 else 0.0
            gap_vars.append(gap_var)
            
            # Linguistic profile extraction from calibration
            window_text = " ".join(w["word"] for w in words_in_window).strip()
            tokens = word_tokenize(window_text)
            tokens_alphanum = [t.lower() for t in tokens if t.isalnum()]
            tagged = pos_tag(tokens)
            
            # TTR
            if tokens_alphanum:
                ttr = len(set(tokens_alphanum)) / (len(tokens_alphanum) + 0.1)
            else:
                ttr = 0.0
            ttr_vals.append(ttr)
            
            # NVR
            nouns_count = len([w for w, t in tagged if t.startswith("NN")])
            verbs_count = len([w for w, t in tagged if t.startswith("VB")])
            nvr = nouns_count / (verbs_count + 0.1) if (nouns_count or verbs_count) else 0.0
            nvr_vals.append(nvr)
            
            # CFR
            content_tags = ("NN", "VB", "JJ", "RB")
            function_tags = ("PRP", "WP", "IN", "DT", "CC", "MD", "TO")
            content_count = len([w for w, t in tagged if t.startswith(content_tags)])
            function_count = len([w for w, t in tagged if t.startswith(function_tags)])
            cfr = content_count / (function_count + 0.1) if (content_count or function_count) else 0.0
            cfr_vals.append(cfr)

            # Filler word detection
            fillers_in_window = 0
            single_word_fillers = {"um", "uh", "er", "like", "so", "basically", "literally", "right"}
            for t in tokens_alphanum:
                if t in single_word_fillers:
                    fillers_in_window += 1
            lower_text = window_text.lower()
            fillers_in_window += lower_text.count("you know")
            fillers_in_window += lower_text.count("i mean")
            filler_rate = fillers_in_window / max(len(tokens_alphanum), 1)
            filler_rate_vals.append(filler_rate)

            # Compute pause entropy
            if gaps:
                bins = np.linspace(0, 2.0, 10)
                counts, _ = np.histogram(gaps, bins=bins)
                probs = counts / (sum(counts) + 1e-6)
                probs = probs[probs > 0]
                pause_entropy = float(-np.sum(probs * np.log2(probs)))
            else:
                pause_entropy = 0.0
            pause_ent_vals.append(pause_entropy)
            
        whisper_stats = {
            "wps_mean": float(np.mean(wps_vals)) if wps_vals else 3.0,
            "wps_std": float(np.std(wps_vals)) or 0.5,
            "gap_var_mean": float(np.mean(gap_vars)) if gap_vars else 0.1,
            "gap_var_std": float(np.std(gap_vars)) or 0.05,
            "filler_mean": float(np.mean(filler_rate_vals)) if filler_rate_vals else 0.0,
            "filler_std": float(np.std(filler_rate_vals)) or 0.01,
            "pause_ent_mean": float(np.mean(pause_ent_vals)) if pause_ent_vals else 0.0,
            "pause_ent_std": float(np.std(pause_ent_vals)) or 0.01
        }
        
        linguistic_baseline = {
            "ttr_mean": float(np.mean(ttr_vals)) if ttr_vals else 0.8,
            "ttr_std": float(np.std(ttr_vals)) or 0.1,
            "nvr_mean": float(np.mean(nvr_vals)) if nvr_vals else 2.0,
            "nvr_std": float(np.std(nvr_vals)) or 0.5,
            "cfr_mean": float(np.mean(cfr_vals)) if cfr_vals else 2.0,
            "cfr_std": float(np.std(cfr_vals)) or 0.5
        }
        
        return {
            "whisper_stats": whisper_stats,
            "parselmouth_baseline": parselmouth_baseline,
            "linguistic_baseline": linguistic_baseline,
            "status": "ok"
        }
    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})
    finally:
        if tmp_path and os.path.exists(tmp_path):
            try: os.unlink(tmp_path)
            except Exception: pass
        if wav_path and os.path.exists(wav_path):
            try: os.unlink(wav_path)
            except Exception: pass
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# -------------------------------------------------------------
# D. Analyze Batch Endpoint (Reconstructed from update_ipynb_batch)
# -------------------------------------------------------------
@app.post("/analyze_batch")
async def analyze_batch(
    audio_file: UploadFile = File(...),
    windows_json: str = Form(...),
    parselmouth_baseline: str = Form(...),
    secret: str = Form(None),
    x_sentinel_secret: str = Header(None)
):
    validate_sentinel_secret(secret, x_sentinel_secret)
    
    t0 = time.perf_counter()
    tmp_path = None
    audio_wav_path = None
    try:
        orig_ext = os.path.splitext(audio_file.filename or ".wav")[1] or ".wav"
        with tempfile.NamedTemporaryFile(delete=False, suffix=orig_ext) as tmp:
            tmp.write(await audio_file.read())
            tmp_path = tmp.name
            
        audio_wav_path = tmp_path + "_analyze.wav"
        ff_res = subprocess.run(
            ["ffmpeg", "-y", "-i", tmp_path, "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", audio_wav_path],
            capture_output=True, text=True
        )
        if ff_res.returncode != 0:
            raise RuntimeError(f"ffmpeg extraction failed: {ff_res.stderr}")
            
        import audiofile
        audio_data, sr = audiofile.read(audio_wav_path)
        if len(audio_data.shape) > 1:
            audio_data = np.mean(audio_data, axis=0)
        if sr != 16000:
            waveform_t = torch.from_numpy(audio_data).float().unsqueeze(0)
            waveform_t = torchaudio.functional.resample(waveform_t, sr, 16000)
            audio_data = waveform_t.squeeze(0).numpy()
            sr = 16000
            
        # Extract baseline profile
        baseline_data = json.loads(parselmouth_baseline)
        if "parselmouth_baseline" in baseline_data:
            baseline = baseline_data["parselmouth_baseline"]
            whisper_stats = baseline_data.get("whisper_stats", {})
            linguistic_baseline = baseline_data.get("linguistic_baseline", {})
        else:
            baseline = baseline_data
            whisper_stats = {}
            linguistic_baseline = {}
            
        # 1. WhisperX ASR on the whole audio file
        result = asr_model.transcribe(audio_data.astype(np.float32), batch_size=2, language="en")
        segments = result.get("segments", [])
        
        all_words_info = []
        if segments:
            aligned_result = whisperx.align(segments, align_model, align_metadata, audio_data.astype(np.float32), device=DEVICE, return_char_alignments=False)
            for seg in aligned_result.get("segments", []):
                for w in seg.get("words", []):
                    all_words_info.append(w)
                    
        timed_words = [w for w in all_words_info if "start" in w and "end" in w]
        
        # Parse windows
        windows = json.loads(windows_json)
        
        # First pass: calculate WPS for all windows to get session pacing metrics
        all_wps = []
        for w_info in windows:
            w_start = w_info["start"]
            w_end = w_start + 4.0
            words_in_window = [w for w in timed_words if w_start <= w["start"] <= w_end]
            all_wps.append(len(words_in_window) / 4.0)
            
        mean_wps_all = np.mean(all_wps) if all_wps else 0.0
        std_wps_all = np.std(all_wps) if all_wps else 0.0
        wps_cv = float(std_wps_all / (mean_wps_all + 1e-6))
        s_pacing_stability = float(np.exp(-0.5 * (wps_cv / 0.15) ** 2))

        # Sentence length uniformity over all segments in the session
        segment_lens = [len(seg.get("text", "").split()) for seg in segments if seg.get("text", "").strip()]
        if len(segment_lens) > 1:
            sent_len_cv = float(np.std(segment_lens) / (np.mean(segment_lens) + 1e-6))
            s_uniformity = float(np.exp(-0.5 * (sent_len_cv / 0.4) ** 2))
        else:
            s_uniformity = 1.0
        
        # Initialize DualProfileEngine and TransitionDetector
        engine = DualProfileEngine(baseline_data)
        transition_detector = TransitionDetector(window_size=5)
        
        batch_results = []
        
        for w_info in windows:
            w_start = w_info["start"]
            w_end = w_start + 4.0
            
            # Slice audio for Parselmouth features (4-second chunk)
            start_sample = int(w_start * sr)
            end_sample = start_sample + int(4.0 * sr)
            if end_sample > len(audio_data):
                chunk = np.pad(audio_data[start_sample:], (0, max(0, end_sample - len(audio_data))))
            else:
                chunk = audio_data[start_sample:end_sample]
                
            chunk_feats = extract_parselmouth_features(chunk, sr)
            
            # Filter words for this window
            words_in_window = [w for w in timed_words if w_start <= w["start"] <= w_end]
            window_text = " ".join(w["word"] for w in words_in_window).strip()
            
            # Tokenize and tag POS
            tokens = word_tokenize(window_text)
            tokens_alphanum = [t.lower() for t in tokens if t.isalnum()]
            tagged = pos_tag(tokens)
            
            # 1. Lexical Diversity (Type-Token Ratio - TTR)
            if tokens_alphanum:
                unique_words = len(set(tokens_alphanum))
                total_words = len(tokens_alphanum)
                ttr = unique_words / (total_words + 0.1)
            else:
                ttr = 0.0
            
            # 2. POS Noun-to-Verb density
            nouns_count = len([w for w, t in tagged if t.startswith("NN")])
            verbs_count = len([w for w, t in tagged if t.startswith("VB")])
            if nouns_count or verbs_count:
                nvr = nouns_count / (verbs_count + 0.1)
            else:
                nvr = 0.0
            
            # 3. Information Density (CFR)
            content_tags = ("NN", "VB", "JJ", "RB")
            function_tags = ("PRP", "WP", "IN", "DT", "CC", "MD", "TO")
            content_count = len([w for w, t in tagged if t.startswith(content_tags)])
            function_count = len([w for w, t in tagged if t.startswith(function_tags)])
            if content_count or function_count:
                cfr = content_count / (function_count + 0.1)
            else:
                cfr = 0.0
                
            # Filler word detection for this window
            fillers_in_window = 0
            single_word_fillers = {"um", "uh", "er", "like", "so", "basically", "literally", "right"}
            for t in tokens_alphanum:
                if t in single_word_fillers:
                    fillers_in_window += 1
            lower_text = window_text.lower()
            fillers_in_window += lower_text.count("you know")
            fillers_in_window += lower_text.count("i mean")
            filler_rate = fillers_in_window / max(len(tokens_alphanum), 1)
            
            # 4. Self-Corrections
            self_corrections = 0
            self_correction_keywords = {"actually", "sorry", "wait", "mean", "i mean", "let me", "start over", "no wait"}
            for kw in self_correction_keywords:
                self_corrections += lower_text.count(kw)
                
            # 5. Retrieval Pauses & Pause Entropy
            gaps = []
            for j in range(len(words_in_window) - 1):
                gap = words_in_window[j+1]["start"] - words_in_window[j]["end"]
                if gap >= 0.0:
                    gaps.append(gap)
            gap_variance = float(np.var(gaps)) if len(gaps) > 1 else 0.0
            
            retrieval_pauses = sum(1 for g in gaps if 0.6 <= g <= 2.5)
            
            if gaps:
                bins = np.linspace(0, 2.0, 10)
                counts, _ = np.histogram(gaps, bins=bins)
                probs = counts / (sum(counts) + 1e-6)
                probs = probs[probs > 0]
                pause_entropy = float(-np.sum(probs * np.log2(probs)))
            else:
                pause_entropy = 0.0

            # Construct the features dictionary
            feats = {
                "pitch_mean": chunk_feats.get("pitch_mean"),
                "pitch_range": chunk_feats.get("pitch_range"),
                "jitter": chunk_feats.get("jitter"),
                "shimmer": chunk_feats.get("shimmer"),
                "hnr": chunk_feats.get("hnr"),
                "wps": float(len(words_in_window) / 4.0),
                "gap_variance": float(gap_variance),
                "ttr": float(ttr),
                "nvr": float(nvr),
                "cfr": float(cfr),
                "filler_rate": float(filler_rate),
                "pitch_accel": chunk_feats.get("pitch_accel", 0.0),
                "energy_fluctuation": chunk_feats.get("energy_fluctuation", 0.0),
                "pause_entropy": float(pause_entropy),
                "self_corrections": float(self_corrections),
                "retrieval_pauses": float(retrieval_pauses)
            }
            
            # Process using DualProfileEngine
            engine_res = engine.process_window(feats)
            
            # Process using TransitionDetector
            trans_res = transition_detector.update(
                script_similarity=engine_res["script_similarity"],
                naturality_score=engine_res["naturality_score"],
                feats=feats
            )
            
            # Parselmouth breakdown for compatibility
            parselmouth_breakdown = {}
            for key in ["jitter", "hnr", "pitch_range", "shimmer"]:
                if key in baseline and key in chunk_feats:
                    b_mean = baseline[key]["mean"]
                    b_std = baseline[key]["std"]
                    val = chunk_feats[key]
                    z = abs(val - b_mean) / b_std
                    parselmouth_breakdown[key] = float(np.exp(-0.5 * z ** 2))
                else:
                    parselmouth_breakdown[key] = 0.0
                    
            batch_results.append({
                "filler_rate_per_30s": float(filler_rate),
                "mean_wps": float(len(words_in_window) / 4.0),
                "wps_cv": float(wps_cv),
                "gap_variance": float(gap_variance),
                "parselmouth_similarity": float(engine_res["script_similarity"]),
                "parselmouth_breakdown": parselmouth_breakdown,
                "gpu_raw_score": float(engine_res["gpu_raw_score"]),
                "transcript": window_text,
                "ttr": float(ttr),
                "nvr": float(nvr),
                "cfr": float(cfr),
                "wps_similarity": float(engine_res["script_similarity"]),
                "wps_stability": float(s_pacing_stability),
                "pacing_score": float(engine_res["script_similarity"]),
                "linguistic_score": float(engine_res["script_similarity"]),
                
                # New Dual-Profile Engine metrics
                "script_similarity": float(engine_res["script_similarity"]),
                "natural_similarity": float(engine_res["natural_similarity"]),
                "naturality_score": float(engine_res["naturality_score"]),
                "transition_detected": bool(trans_res["transition_detected"]),
                "transition_confidence": float(trans_res["confidence"]),
                
                # Dynamic naturality features
                "pitch_accel": float(feats["pitch_accel"]),
                "energy_fluctuation": float(feats["energy_fluctuation"]),
                "pause_entropy": float(feats["pause_entropy"]),
                "self_corrections": float(feats["self_corrections"]),
                "retrieval_pauses": float(feats["retrieval_pauses"])
            })
            
        processing_time_ms = int((time.perf_counter() - t0) * 1000.0)
        
        return {
            "results": batch_results,
            "processing_time_ms": processing_time_ms,
            "status": "ok"
        }
    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})
    finally:
        if tmp_path and os.path.exists(tmp_path):
            try: os.unlink(tmp_path)
            except Exception: pass
        if audio_wav_path and os.path.exists(audio_wav_path):
            try: os.unlink(audio_wav_path)
            except Exception: pass
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# -------------------------------------------------------------
# E. Start ngrok tunnel and print the public URL
# -------------------------------------------------------------
nest_asyncio.apply()
port = 8000

# Clear any existing tunnels
ngrok.kill()

print("Starting ngrok tunnel...")
public_url = ngrok.connect(port).public_url
print(f"SENTINEL GPU SERVER LIVE: {public_url}")

config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="info")
server = uvicorn.Server(config)
import asyncio
loop = asyncio.get_event_loop()
loop.run_until_complete(server.serve())
